# Text Classification

> *right now we will be learning text classification for ML, it will not be deep learning based text classification*

There are different types of text classification:

1. binary classification
2. multiclass classification
3. multi label classification

In [1]:
%load_ext cudf.pandas

In [2]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('/kaggle/input/datasets/ashura369/imdb-dataset/IMDB Dataset.csv', engine='python', on_bad_lines='skip')
df.head(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [4]:
print(df.shape)
print(df.isna().sum())
print()
print(f"DUPLICATED SAMPLES : {df.duplicated().sum()}")

# dropping the duplicates
df = df.drop_duplicates()
print(f"DUPLICATED SAMPLES : {df.duplicated().sum()}")


(50000, 2)
review       0
sentiment    0
dtype: int64

DUPLICATED SAMPLES : 418
DUPLICATED SAMPLES : 0


In [5]:
df.shape

(49582, 2)

In [6]:
# data cleaning -- removing html tags using regex,
# you can also use 'beautifulsoup', but it will be slow on huge datasets

import re

# df['transformed'] = df['review'].apply(lambda x : re.sub(r'<[^>]+>', '', x))
# you can also use the code above

text = "<p>This is a <b>great</b> movie!<br /> Highly recommend.</p>"
re.sub(r'<[^>]+>', '', text)



'This is a great movie! Highly recommend.'

In [7]:
!pip install cleantext

In [8]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
stop_words = stopwords.words('english')

from nltk.stem import WordNetLemmatizer
lmt = WordNetLemmatizer()

from cleantext import clean

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
def transform(txt):
  # removing html tags
  txt = re.sub(r'<[^>]+>', '', txt)

  # lowercase and removing punctuations
  txt = clean(txt, lowercase=True, punct=True)

  # tokenization
  txt = nltk.word_tokenize(txt)

  temp = [lmt.lemmatize(word, pos='v') for word in txt if word not in stop_words]

  temp2 = temp[:]
  temp2 = " ".join(temp2)

  return temp2

In [10]:
transform('Hello this the God King, playing and singing songs of liberation')

'hello god king play sing songs liberation'

In [11]:
df['transformed'] = df['review'].apply(transform)
df.head(10)

,review,sentiment,transformed
0,One of the other reviewers has mentioned that ...,positive,one reviewers mention watch 1 oz episode youll...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,positive,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake think ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stun f...
5,"Probably my all-time favorite movie, a story o...",positive,probably alltime favorite movie story selfless...
6,I sure would like to see a resurrection of a u...,positive,sure would like see resurrection date seahunt ...
7,"This show was an amazing, fresh & innovative i...",negative,show amaze fresh innovative idea 70s first air...
8,Encouraged by the positive comments about this...,negative,encourage positive comment film look forward w...
9,If you like original gut wrenching laughter yo...,positive,like original gut wrench laughter like movie y...


In [12]:
# using label encoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [13]:
df['sentiment'] = le.fit_transform(df['sentiment'])

In [14]:
for i, j in enumerate(le.classes_):
  print(f"THE NUMBER {i} REPRESENTS : {j}")

THE NUMBER 0 REPRESENTS : negative
THE NUMBER 1 REPRESENTS : positive


In [15]:
df.head()

,review,sentiment,transformed
0,One of the other reviewers has mentioned that ...,1,one reviewers mention watch 1 oz episode youll...
1,A wonderful little production. <br /><br />The...,1,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,1,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,0,basically theres family little boy jake think ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,petter matteis love time money visually stun f...


In [16]:
x = df['transformed']
y = df['sentiment'].values

In [17]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [18]:
# vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)      # always use max_features, else the code will crash

In [19]:
x_train_tfidf = tfidf.fit_transform(x_train).toarray()
x_test_tfidf = tfidf.transform(x_test).toarray()

In [20]:
x_train_tfidf

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [21]:
x_test_tfidf

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.06778908, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.04941163, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

### using Naive Bayes for prediction

In [22]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()

In [23]:
from sklearn.metrics import accuracy_score, confusion_matrix

In [28]:
nb.fit(x_train_tfidf, y_train)
y_pred = nb.predict(x_test_tfidf)

In [32]:
df.shape

(49582, 3)

In [29]:
print(f"ACCURACY SCORE : {accuracy_score(y_test, y_pred)}")

ACCURACY SCORE : 0.7977142857142857


In [31]:
print(f"CONFUSION MATRIX : \n {confusion_matrix(y_test, y_pred)}")

CONFUSION MATRIX : 
 [[5924 1480]
 [1529 5942]]


### using RandomForestClassifier for prediction

In [34]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [35]:
rf.fit(x_train_tfidf, y_train)
y_pred = rf.predict(x_test_tfidf)

In [38]:
print(f"ACCURACY SCORE : {accuracy_score(y_test, y_pred)}")
print(f"\n CONFUSION MATRIX : \n{confusion_matrix(y_test, y_pred)}")

ACCURACY SCORE : 0.8420840336134454

 CONFUSION MATRIX : 
[[6282 1122]
 [1227 6244]]


## **IMP**

**While using text classification, it is recommended to use Word2Vec.**

There are two ways to use it, either you can use an already available dataset like google news, or you can also use the dataset you are already working on.

- Pros : google news dataset is huge, large number of words of vacabulary | use it only when your dataset is small
- Cons : looses the sentence contextuality, if you use your own dataset, the model will be built on the contextual meaning of your own dataset

If you want to use google news dataset, then make sure that **80 % OF WORDS VOCABULARY IN YOUR DATASET IS AVAILABLE IN GOOGLE NEWS DATASET**. Else the model will not work properly.